# Athena Audit Dashboard

Interactive notebook for querying Athena audit data (IDC workgroups).

1. Run **Cell 2** (authenticate) and **Cell 3** (connect)
2. Edit the **Filters** in Cell 4, then run the action cells below

Requirements: `pip install pyathena pandas plotly cihi_auth`

In [ ]:
# === Step 1: IDC Authentication ===
from cihi_auth.jupyter_helper import authenticate, get_session

authenticate()
session = get_session(profile='default')
print('Authenticated via TIP')

In [ ]:
# === Step 2: Connect to Athena ===
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import plotly.express as px
from pyathena import connect
from IPython.display import display

REGION = 'us-east-1'
DATABASE = 'athena_events'
TABLE = 'events'
WORKGROUP = 'idc-wg'

# Athena pricing - adjust if your pricing differs
COST_PER_TB = 5.00  # USD per TB scanned (change to match your rate)

creds = session.get_credentials().get_frozen_credentials()
conn = connect(
    region_name=REGION,
    work_group=WORKGROUP,
    schema_name=DATABASE,
    aws_access_key_id=creds.access_key,
    aws_secret_access_key=creds.secret_key,
    aws_session_token=creds.token,
)
print('Connected to Athena (' + WORKGROUP + ')')

---
## Filters
Edit the values below and run the cell. Then run any action cell underneath.

In [ ]:
# === EDIT YOUR FILTERS HERE, then run this cell ===

FROM_DATE = '2026-02-22'   # start date (YYYY-MM-DD)
TO_DATE   = '2026-03-01'   # end date (YYYY-MM-DD)
USER      = ''             # source_identity filter (leave empty for all)
WORKGROUP_FILTER = ''      # workgroup filter (leave empty for all)
STATUS    = ''             # SUCCEEDED, FAILED, CANCELLED (leave empty for all)
QUERY_CONTAINS = ''        # text search in query (leave empty for all)
MAX_ROWS  = 200            # max rows to return

# --- Build WHERE clause (do not edit below) ---
def _where():
    c = ["day BETWEEN '" + FROM_DATE + "' AND '" + TO_DATE + "'"]
    if USER:
        c.append("source_identity = '" + USER + "'")
    if WORKGROUP_FILTER:
        c.append("workgroup = '" + WORKGROUP_FILTER + "'")
    if STATUS:
        c.append("status = '" + STATUS + "'")
    if QUERY_CONTAINS:
        c.append("LOWER(query) LIKE '%" + QUERY_CONTAINS.lower() + "%'")
    return ' AND '.join(c)

print('Filters set:')
print('  Date range: ' + FROM_DATE + ' to ' + TO_DATE)
print('  User:       ' + (USER or '(all)'))
print('  Workgroup:  ' + (WORKGROUP_FILTER or '(all)'))
print('  Status:     ' + (STATUS or '(all)'))
print('  Query text: ' + (QUERY_CONTAINS or '(any)'))
print('  Max rows:   ' + str(MAX_ROWS))

---
## Available Users
Run this cell to see what users/workgroups exist in the data.

In [ ]:
# === List available users and workgroups ===
users = pd.read_sql(
    "SELECT source_identity, COUNT(*) AS queries "
    "FROM " + TABLE + " "
    "WHERE source_identity IS NOT NULL "
    "AND day BETWEEN '" + FROM_DATE + "' AND '" + TO_DATE + "' "
    "GROUP BY 1 ORDER BY 2 DESC", conn
)
wgs = pd.read_sql(
    "SELECT workgroup, COUNT(*) AS queries "
    "FROM " + TABLE + " "
    "WHERE workgroup IS NOT NULL "
    "AND day BETWEEN '" + FROM_DATE + "' AND '" + TO_DATE + "' "
    "GROUP BY 1 ORDER BY 2 DESC", conn
)
print('=== USERS ===')
display(users)
print()
print('=== WORKGROUPS ===')
display(wgs)

---
## User Queries (Primary View)
Shows queries run by the selected user with status and query text.
Set `USER` in the Filters cell above, then run this cell.

In [ ]:
# === USER QUERIES ===
sql = (
    'SELECT event_time, source_identity, workgroup, status, '
    'SUBSTR(query, 1, 300) AS query_text, '
    '"database", ROUND(data_scanned / 1048576.0, 2) AS data_mb, '
    'ROUND(cost, 6) AS cost_usd '
    'FROM ' + TABLE + ' '
    'WHERE ' + _where() + ' '
    'ORDER BY event_time DESC '
    'LIMIT ' + str(MAX_ROWS)
)

df = pd.read_sql(sql, conn)

if df.empty:
    print('No results. Try adjusting filters.')
else:
    print(str(len(df)) + ' queries found')
    print()
    # Status summary
    summary = df.groupby('status').agg(
        count=('status', 'size'),
        total_mb=('data_mb', 'sum'),
        total_cost=('cost_usd', 'sum')
    ).reset_index()
    print('--- Status Summary ---')
    for _, r in summary.iterrows():
        print('  ' + str(r['status']) + ': ' + str(int(r['count'])) + ' queries, ' + str(round(r['total_mb'], 2)) + ' MB, $' + str(round(r['total_cost'], 4)))
    total_cost = df['cost_usd'].sum()
    print()
    print('  TOTAL COST: $' + str(round(total_cost, 4)))
    print()
    display(df)

In [ ]:
# === EXPORT last query results to CSV ===
if 'df' in dir() and df is not None and not df.empty:
    df.to_csv('audit_export.csv', index=False)
    print('Exported ' + str(len(df)) + ' rows to audit_export.csv')
else:
    print('No data to export. Run User Queries first.')

---
## Dashboard
Summary statistics and charts for the selected filters.

In [ ]:
# === SUMMARY STATS ===
w = _where()
s = pd.read_sql(
    'SELECT COUNT(*) AS total, '
    'COUNT(DISTINCT source_identity) AS users, '
    'COUNT(DISTINCT workgroup) AS workgroups, '
    "SUM(CASE WHEN status='SUCCEEDED' THEN 1 ELSE 0 END) AS ok, "
    "SUM(CASE WHEN status='FAILED' THEN 1 ELSE 0 END) AS fail, "
    "SUM(CASE WHEN status='CANCELLED' THEN 1 ELSE 0 END) AS cancel, "
    'ROUND(SUM(data_scanned)/1073741824.0,3) AS gb, '
    'ROUND(SUM(cost), 4) AS total_cost '
    'FROM ' + TABLE + ' WHERE ' + w, conn
).iloc[0]

print('=' * 55)
print('  AUDIT SUMMARY: ' + FROM_DATE + ' to ' + TO_DATE)
print('=' * 55)
print('  Total Queries:   ' + str(int(s['total'])))
print('  Unique Users:    ' + str(int(s['users'])))
print('  Workgroups:      ' + str(int(s['workgroups'])))
print('  Succeeded:       ' + str(int(s['ok'])))
print('  Failed:          ' + str(int(s['fail'])))
print('  Cancelled:       ' + str(int(s['cancel'])))
print('  Data Scanned:    ' + str(s['gb']) + ' GB')
print('  Total Cost:      $' + str(s['total_cost']))
print('=' * 55)

In [ ]:
# === QUERIES PER USER by Status ===
udf = pd.read_sql(
    'SELECT COALESCE(source_identity, user_identity_type) AS user_name, '
    'status, COUNT(*) AS cnt '
    'FROM ' + TABLE + ' WHERE ' + _where() + ' '
    'GROUP BY 1, 2 ORDER BY cnt DESC LIMIT 50', conn
)
if not udf.empty:
    fig = px.bar(udf, x='user_name', y='cnt', color='status',
                 title='Queries Per User by Status',
                 labels={'user_name': 'User', 'cnt': 'Count', 'status': 'Status'},
                 color_discrete_map={'SUCCEEDED': '#2ecc71', 'FAILED': '#e74c3c', 'CANCELLED': '#f39c12'},
                 barmode='stack')
    fig.update_layout(xaxis_tickangle=-45, height=400)
    fig.show()
else:
    print('No data for chart')

In [ ]:
# === STATUS BREAKDOWN ===
sdf = pd.read_sql(
    "SELECT COALESCE(status, 'UNKNOWN') AS status, COUNT(*) AS cnt "
    'FROM ' + TABLE + ' WHERE ' + _where() + ' '
    'GROUP BY 1', conn
)
if not sdf.empty:
    fig = px.pie(sdf, names='status', values='cnt', title='Status Breakdown',
                 color='status',
                 color_discrete_map={'SUCCEEDED': '#2ecc71', 'FAILED': '#e74c3c', 'CANCELLED': '#f39c12', 'UNKNOWN': '#95a5a6'})
    fig.update_traces(textinfo='label+percent+value')
    fig.update_layout(height=400)
    fig.show()
else:
    print('No data for chart')

In [ ]:
# === QUERIES PER DAY ===
tdf = pd.read_sql(
    'SELECT day, workgroup, COUNT(*) AS cnt '
    'FROM ' + TABLE + ' WHERE ' + _where() + ' '
    'GROUP BY 1, 2 ORDER BY 1', conn
)
if not tdf.empty:
    fig = px.bar(tdf, x='day', y='cnt', color='workgroup',
                 title='Queries Per Day by Workgroup', barmode='stack',
                 labels={'day': 'Day', 'cnt': 'Count', 'workgroup': 'Workgroup'})
    fig.update_layout(height=400)
    fig.show()
else:
    print('No data for chart')

In [ ]:
# === DATA SCANNED & COST PER WORKGROUP ===
ddf = pd.read_sql(
    'SELECT workgroup, COUNT(*) AS cnt, '
    'ROUND(SUM(data_scanned)/1073741824.0, 3) AS gb, '
    'ROUND(SUM(cost), 4) AS cost_usd '
    'FROM ' + TABLE + ' WHERE ' + _where() + ' AND workgroup IS NOT NULL '
    'GROUP BY 1 ORDER BY cost_usd DESC', conn
)
if not ddf.empty:
    print('Cost per workgroup:')
    for _, r in ddf.iterrows():
        print('  ' + str(r['workgroup']) + ': ' + str(int(r['cnt'])) + ' queries, ' + str(r['gb']) + ' GB, $' + str(r['cost_usd']))
    print()
    fig = px.bar(ddf, x='workgroup', y='cost_usd', text='cnt',
                 title='Cost Per Workgroup (USD)',
                 labels={'workgroup': 'Workgroup', 'cost_usd': 'Cost ($)', 'cnt': 'Queries'},
                 color='cost_usd', color_continuous_scale='Reds')
    fig.update_traces(texttemplate='%{text} queries', textposition='outside')
    fig.update_layout(height=400)
    fig.show()
else:
    print('No data for chart')

In [ ]:
# === COST PER USER ===
cdf = pd.read_sql(
    'SELECT COALESCE(source_identity, user_identity_type) AS user_name, '
    'COUNT(*) AS queries, '
    'ROUND(SUM(data_scanned)/1073741824.0, 3) AS gb, '
    'ROUND(SUM(cost), 4) AS cost_usd '
    'FROM ' + TABLE + ' WHERE ' + _where() + ' '
    'GROUP BY 1 ORDER BY cost_usd DESC LIMIT 30', conn
)
if not cdf.empty:
    print('Top users by cost:')
    for _, r in cdf.iterrows():
        print('  ' + str(r['user_name']) + ': ' + str(int(r['queries'])) + ' queries, ' + str(r['gb']) + ' GB, $' + str(r['cost_usd']))
    print()
    fig = px.bar(cdf, x='user_name', y='cost_usd', text='queries',
                 title='Cost Per User (USD)',
                 labels={'user_name': 'User', 'cost_usd': 'Cost ($)', 'queries': 'Queries'},
                 color='cost_usd', color_continuous_scale='Blues')
    fig.update_traces(texttemplate='%{text} queries', textposition='outside')
    fig.update_layout(xaxis_tickangle=-45, height=400)
    fig.show()
else:
    print('No data for chart')

---
## Ad-Hoc SQL
Modify the SQL below and run the cell.

In [ ]:
# === Custom SQL ===
custom_sql = """
SELECT source_identity, workgroup, status, COUNT(*) AS cnt,
       ROUND(SUM(data_scanned) / 1048576.0, 2) AS total_mb,
       ROUND(SUM(cost), 4) AS total_cost_usd
FROM events
WHERE day >= '2026-02-28'
  AND source_identity IS NOT NULL
GROUP BY source_identity, workgroup, status
ORDER BY cnt DESC
LIMIT 50
"""

result = pd.read_sql(custom_sql, conn)
display(result)